# KDP-TMA — Week 1 Quick Start

**Goal:** Set up the environment, partition the data, and run the FedAvg baseline on Fashion-MNIST to a final accuracy of ~85%.

**Estimated GPU time:** ~1 hour on Colab Pro (T4 or A100).

**Before you start:** switch runtime to GPU (Runtime → Change runtime type → GPU).

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone your GitHub repo

Replace `YOUR_USERNAME` with your GitHub handle.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/kdp-tma-federated.git
%cd kdp-tma-federated

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Partition Fashion-MNIST with Dirichlet(α=0.5) over 100 clients

In [ ]:
!python data/partition.py --dataset fashion-mnist --clients 100 --alpha 0.5

## 5. Visual sanity check — one client's class distribution

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

with open('partitions/fashion-mnist_alpha0.5_n100.pkl', 'rb') as f:
    partition = pickle.load(f)

ds = datasets.FashionMNIST('./data_cache', train=True, download=False)
labels = np.array(ds.targets)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, cid in zip(axes, [0, 1, 2]):
    client_labels = labels[partition[cid]]
    counts = np.bincount(client_labels, minlength=10)
    ax.bar(range(10), counts)
    ax.set_title(f'Client {cid} ({len(partition[cid])} samples)')
    ax.set_xlabel('Class')
plt.tight_layout()
plt.show()

print('Non-IID confirmed: each client has an uneven class distribution.')

## 6. Quick sanity run — 20 rounds only

Runs in ~5 minutes. Confirms the pipeline works before committing to the full 200-round run.

In [ ]:
!python scripts/main.py \
    --dataset fashion-mnist \
    --defense fedavg \
    --attack none \
    --rounds 20 \
    --seed 42

**Expected after 20 rounds:** accuracy around 65-75%. If lower, something is wrong — check the printed configuration and verify the partition file exists.

## 7. Full run — 200 rounds

Takes ~1 hour on T4. Run only after step 6 succeeds.

In [ ]:
!python scripts/main.py \
    --dataset fashion-mnist \
    --defense fedavg \
    --attack none \
    --rounds 200 \
    --seed 42

**Expected after 200 rounds:** accuracy around 85-88%.

## 8. Save logs to Google Drive (persistence across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/kdp_tma_logs
!cp -r logs/* /content/drive/MyDrive/kdp_tma_logs/
print('Logs saved to Drive.')

## 9. Push to GitHub

In [ ]:
!git add logs/ partitions/
!git -c user.email='jenan.jader@atu.edu.iq' -c user.name='Jenan Jader' \
   commit -m 'Week 1: FedAvg baseline complete on Fashion-MNIST'
!git push

---

## Week 1 Checklist

- [ ] GPU available on Colab Pro (T4 or A100)
- [ ] Repository cloned and dependencies installed
- [ ] Dirichlet partitioning produces non-IID splits (visualisation confirms)
- [ ] 20-round sanity check reaches ~70% accuracy
- [ ] Full 200-round run reaches ≥85% accuracy
- [ ] Logs saved to Google Drive
- [ ] Code committed to GitHub

**When all boxes are ticked, message me for the Week 2 code.**